In [0]:
# Databricks notebook source
# Gold Layer - fact_sales
# NOTE: run 02_dim_customer and 03_dim_product (and 01_dim_date, whenever the
# data's date range changes) BEFORE this notebook, so every CustomerID/StockCode
# in the current Silver batch already has a surrogate key to join to.

from pyspark.sql import functions as F

# COMMAND ----------

# Structured Streaming source off the Silver Delta table. Delta tables can be
# read as a stream, so this picks up only the new/changed rows since the last
# checkpointed run - consistent with the Bronze/Silver Auto Loader pattern.
silver_stream = spark.readStream.table("retail_sales_dev.silver.sales_cleaned")

# Dimensions are read as static (batch) DataFrames. A streaming-to-static join
# is fully supported by Structured Streaming and does not require foreachBatch,
# because only one side of the join is a streaming source.
dim_customer = spark.table("retail_sales_dev.gold.dim_customer").select(
    "CustomerID", "CustomerSK"
)
dim_product = spark.table("retail_sales_dev.gold.dim_product").select(
    "StockCode", "ProductSK"
)
dim_date = spark.table("retail_sales_dev.gold.dim_date").select(
    "CalendarDate", "DateSK"
)


In [0]:

# COMMAND ----------

fact_df = (
    silver_stream
    .join(dim_customer, on="CustomerID", how="left")
    .join(dim_product, on="StockCode", how="left")
    .join(
        dim_date,
        F.to_date(silver_stream.InvoiceDate) == dim_date.CalendarDate,
        how="left",
    )
    .select(
        F.col("Invoice").alias("InvoiceNumber"),
        F.col("CustomerSK"),
        F.col("ProductSK"),
        F.col("DateSK"),
        F.col("Quantity"),
        F.col("Price").alias("UnitPrice"),
        (F.col("Quantity") * F.col("Price")).alias("TotalAmount"),
        F.col("is_cancelled"),
        F.col("InvoiceDate"),
    )
)

# COMMAND ----------

(
    fact_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/checkpoints/gold_fact_sales/")  # match your existing checkpoint root
    .trigger(availableNow=True)
    .toTable("retail_sales_dev.gold.fact_sales")
)

In [0]:
%sql

Select * From retail_sales_dev.gold.fact_sales